In [1]:
import numpy as np
import torch

In [2]:
PIECE_TO_PLANE = {
    # 0‒11：走子 **之前** 的棋子分层（白大写，黑小写）
    "P": 0, "N": 1, "B": 2, "R": 3, "Q": 4, "K": 5,
    "p": 6, "n": 7, "b": 8, "r": 9, "q":10, "k":11,
    # 12‒23：走子 **之后** 的棋子分层（在这里只用 FEN，因而同 before；如需
    #        “走子前/后”两张局面，可把 before 层留空或单独传入）
    "P_after":12, "N_after":13, "B_after":14, "R_after":15,
    "Q_after":16, "K_after":17,
    "p_after":18, "n_after":19, "b_after":20, "r_after":21,
    "q_after":22, "k_after":23,
}

# 24‒33：10 个元数据平面下标
PLANE_SIDE_TO_MOVE      = 24  # 白走=1，黑走=0
PLANE_CASTLING_WK       = 25  # White K-side castling right
PLANE_CASTLING_WQ       = 26
PLANE_CASTLING_BK       = 27
PLANE_CASTLING_BQ       = 28
PLANE_THREEFOLD_WARN    = 29  # 1 = 局面已重复 ≥2 次
PLANE_FIFTYMOVE_WARN    = 30  # 1 = ≥ 50 half-moves 自上次吃子/兵走
PLANE_EN_PASSANT_FILE   = 31  # 仅把可吃过路兵 file 所有格设 1
PLANE_BORDER            = 32  # 边界提示（外一圈 1，其余 0）
PLANE_EMPTY             = 33  # “占位”或自定义信息（这里全 0）

In [3]:
def fen_to_34ch_tensor(fen: str, before_board: np.ndarray | None = None,
                       threefold_cnt: int = 1, halfmove_clock: int = 0) -> torch.Tensor:
    """
    参数
    ----
    fen            : 标准 FEN 字符串（不含 move no. 亦可）
    before_board   : 可选 8×8 字符数组，表示走子前局面；若缺省则用当前位置填充
    threefold_cnt  : 当前局面出现的次数（≥ 1）；≥ 2 时 Threefold 平面置 1
    halfmove_clock : FEN 字符串里第 5 字段；≥ 100 时 50-move 平面置 1

    返回
    ----
    torch.FloatTensor  shape = (34, 8, 8)
    """
    board_fen, side, castling, ep, hm_clock, *_ = (fen + " - - 0 1").split()[:6]

    # --- 1) 初始化空张量 ---
    planes = np.zeros((34, 8, 8), dtype=np.float32)

    # --- 2) 填棋子平面 (after) ---
    rows = board_fen.split("/")
    for rank, row in enumerate(rows):               # rank: 0 (8th) … 7 (1st)
        file_idx = 0
        for char in row:
            if char.isdigit():
                file_idx += int(char)
                continue
            plane = PIECE_TO_PLANE[char] + 12       # after 层
            planes[plane, rank, file_idx] = 1.0
            # 如提供 before_board，则把 before plane 设到 0-11
            before_plane = plane - 12
            if before_board is None or before_board[rank, file_idx] == char:
                planes[before_plane, rank, file_idx] = 1.0
            file_idx += 1

    # 若给了 before_board，可按需覆写 0-11 层
    if before_board is not None:
        for rank in range(8):
            for file_idx in range(8):
                piece = before_board[rank, file_idx]
                if piece != ".":
                    planes[PIECE_TO_PLANE[piece], rank, file_idx] = 1.0

    # --- 3) Side-to-move ---
    if side == "w":
        planes[PLANE_SIDE_TO_MOVE, :, :] = 1.0

    # --- 4) Castling rights ---
    planes[PLANE_CASTLING_WK, :, :] = 1.0 if "K" in castling else 0.0
    planes[PLANE_CASTLING_WQ, :, :] = 1.0 if "Q" in castling else 0.0
    planes[PLANE_CASTLING_BK, :, :] = 1.0 if "k" in castling else 0.0
    planes[PLANE_CASTLING_BQ, :, :] = 1.0 if "q" in castling else 0.0

    # --- 5) Three-fold repetition / 50-move rule ---
    if threefold_cnt >= 2:
        planes[PLANE_THREEFOLD_WARN, :, :] = 1.0
    if int(halfmove_clock) >= 100 or halfmove_clock >= 100:
        planes[PLANE_FIFTYMOVE_WARN, :, :] = 1.0

    # --- 6) En-passant file (若存在) ---
    if ep != "-":
        file_char = ep[0]
        file_idx = ord(file_char) - ord("a")
        planes[PLANE_EN_PASSANT_FILE, :, file_idx] = 1.0

    # --- 7) Board border ---
    planes[PLANE_BORDER, 0, :] = planes[PLANE_BORDER, 7, :] = 1.0
    planes[PLANE_BORDER, :, 0] = planes[PLANE_BORDER, :, 7] = 1.0

    return torch.from_numpy(planes)

In [4]:
fen = "r1bqkbnr/pppp1ppp/2n5/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 2 3"
tensor34 = fen_to_34ch_tensor(fen)
print(tensor34.shape)   # torch.Size([34, 8, 8])

torch.Size([34, 8, 8])


In [ ]:
import io
import os
import chess
import bz2
import chess.pgn

fold_path = "./players/"
output_path = "./dataset2/"

def board_to_array(board: chess.Board) -> np.ndarray:
    """返回 8×8 numpy 数组，空格用 '.' 占位，a1 在 [7,0] 位置"""
    arr = np.full((8, 8), ".", dtype="<U1")
    for square, piece in board.piece_map().items():
        rank = 7 - chess.square_rank(square)   # 0 行对应 8 路
        file = chess.square_file(square)       # 0 列对应 a 文件
        arr[rank, file] = piece.symbol()       # 'P','k',...
    return arr

os.makedirs(output_path, exist_ok=True)

for path in os.listdir(fold_path):
    path = os.path.join(fold_path, path)
    for split in ('white', 'black'):
        with bz2.open(f"{path}/{split}.pgn.bz2", mode="rt", encoding="utf-8") as compressed_file:
            pgn_text = compressed_file.read()
            pgn_io = io.StringIO(pgn_text)

            while True:
                game = chess.pgn.read_game(pgn_io)
                if game is None:
                    break

                board = game.board()
                game_tensor = []
                for move in game.mainline_moves():
                    before_mat = board_to_array(board)
                    board.push(move)
                    after_fen = board.fen()
                    position_tensor = fen_to_34ch_tensor(after_fen, before_board=before_mat)
                    game_tensor.append(position_tensor)

                game_tensor = torch.stack(game_tensor, dim=0)
                

torch.Size([11, 34, 8, 8])
torch.Size([41, 34, 8, 8])
torch.Size([33, 34, 8, 8])
torch.Size([69, 34, 8, 8])
torch.Size([51, 34, 8, 8])
torch.Size([62, 34, 8, 8])
torch.Size([51, 34, 8, 8])
torch.Size([41, 34, 8, 8])
torch.Size([63, 34, 8, 8])
torch.Size([39, 34, 8, 8])
torch.Size([25, 34, 8, 8])
torch.Size([46, 34, 8, 8])
torch.Size([82, 34, 8, 8])
torch.Size([56, 34, 8, 8])
torch.Size([55, 34, 8, 8])
torch.Size([66, 34, 8, 8])
torch.Size([47, 34, 8, 8])
torch.Size([54, 34, 8, 8])
torch.Size([53, 34, 8, 8])
torch.Size([53, 34, 8, 8])
torch.Size([45, 34, 8, 8])
torch.Size([27, 34, 8, 8])
torch.Size([61, 34, 8, 8])
torch.Size([69, 34, 8, 8])
torch.Size([52, 34, 8, 8])
torch.Size([76, 34, 8, 8])
torch.Size([66, 34, 8, 8])
torch.Size([53, 34, 8, 8])
torch.Size([69, 34, 8, 8])
torch.Size([59, 34, 8, 8])
torch.Size([81, 34, 8, 8])
torch.Size([57, 34, 8, 8])
torch.Size([66, 34, 8, 8])
torch.Size([53, 34, 8, 8])
torch.Size([61, 34, 8, 8])
torch.Size([35, 34, 8, 8])
torch.Size([49, 34, 8, 8])
t

KeyboardInterrupt: 

: 